# Structure Data Manager

Notebook điều phối crawler vnstock (module `structure_data`): retry/fallback đa nguồn, QC OHLCV, pipeline một lệnh tùy chọn.

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.
**Nguồn:** `primary_source` / `fallback_source` (vd. **VCI** + **KBS**). Key vnstock: tạo file **`stock-pipeline/.env`** với dòng `VNSTOCK_API_KEY=...` (đã có trong `.gitignore`); `register_vnstock_api_key_from_env` tự gọi `load_dotenv`. Có thể dùng `.env.example` làm mẫu.

**Mục đích:** 
- **Giá OHLCV (`price`)**, **chỉ số (`index`)**, **financial ratio (`financial_ratio`)**: lưu theo **`date=<ngày chạy notebook>`**. Mỗi mã (hoặc mỗi mã chỉ số) một file Parquet trong partition đó. **Chưa có** partition/file → tạo mới; **đã có** → **ghi đè**.
- **Company overview**: file master **`company/master/company_overview.parquet`** — cơ chế **append** (mỗi lần chạy thêm batch, có `ingest_run_id`), **không** xóa lịch sử các lần trước.
- **Listing**: snapshot master tại `listing/master/` — mỗi lần chạy **ghi đè** file listing hiện tại.

**Gợi ý:** chạy từng bước (ô dưới) hoặc `run_structure_ingestion_pipeline(cfg)` để nghỉ `delay_between_categories_sec` giữa các nhóm API.

In [1]:
import os
import sys
import warnings
import threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Suppress encoding errors từ subprocess threads
original_hook = threading.excepthook
def custom_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass  # Ignore encoding errors from threads
    else:
        original_hook(args)
threading.excepthook = custom_hook

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

print("[OK] UTF-8 guard enabled for current kernel session")

[OK] UTF-8 guard enabled for current kernel session


In [ ]:
import os
import importlib
import structure_data as sd

# Reload de tranh cache module cu trong notebook kernel
sd = importlib.reload(sd)

from structure_data import (
    IngestionConfig,
    configure_logging,
    register_vnstock_api_key_from_env,
    run_structure_ingestion_pipeline,
    ingest_prices,
    ingest_indices,
    ingest_listing,
    ingest_company_overview,
    ingest_financial_ratio,
    ingest_price_board,
)

register_vnstock_api_key_from_env("VNSTOCK_API_KEY")

configure_logging()
cfg = IngestionConfig()

# Quan ly danh sach ma co phieu ngay tai notebook (them/bot moi lan chay)
cfg.tickers = [
    # VN30 hiện tại (20 mã)
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG",
    "MBB","MSN","MWG","PLX","POW","SAB","SHB","SSI","STB","TCB",
    # # Thêm 30 mã đa ngành
    "VCB","VHM","VIC","VJC","VNM","VPB","VRE","TPB","VIB","HVN",
    "REE","PNJ","GMD","DGC","DPM","DCM","HSG","KDC","QNS","SCS",
    
    "NVL","PDR","DXG","KDH","HDG","EVF","KBC","AGG","TCH","VPI",
]

# Co the sua bo chi so neu can
cfg.index_tickers = ["VNINDEX", "VN30", "HNXINDEX", "HNX30", "UPCOMINDEX"]

cfg.max_tickers_per_run = len(cfg.tickers)
cfg.rate_limit_rpm = 10
cfg.inter_request_delay_sec = 0.5

cfg.primary_source = "vci"
cfg.fallback_source = "kbs"

print('has ingest_price_board:', hasattr(sd, 'ingest_price_board'))
print(cfg)

📦 **Vnstock 3.5.1 is available**
Current: 3.5.0 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.6 is available**
Current: 2.4.0 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
has ingest_price_board: True
IngestionConfig(tickers=['ACB', 'BCM', 'BID', 'BVH', 'CTG', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'MBB', 'MSN', 'MWG', 'PLX', 'POW', 'SAB', 'SHB', 'SSI', 'STB', 'TCB', 'VCB', 'VHM', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE', 'TPB', 'VIB', 'HVN', 'REE', 'PNJ', 'GMD', 'DGC', 'DPM', 'DCM', 'HSG', 'KDC', 'QNS', 'SCS', 'NVL', 'PDR', 'DXG', 'KDH', 'HDG', 'EVF', 'KBC', 'AGG', 'TCH', 'VPI'], index_tickers=['VNINDEX', 'VN30', 'HNXINDEX', 'HNX30', 'UPCOMINDEX'], primary_source='vci', fallback_source='kbs', rate_limit_rpm=5, years_back=5, max_tickers_per_run=50, delay_between_categories_sec=

In [3]:
# 1) OHLCV gia co phieu
price_outputs = ingest_prices(cfg)
len(price_outputs), list(price_outputs.items())[:3]

2026-04-11 00:42:27 [INFO] [1/50] Fetching ACB...
2026-04-11 00:42:29 [INFO] ACB — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-04-11 00:42:29 [INFO] ACB | src_used=VCI rows=1306 min_date=2021-01-12 max_date=2026-04-10 pct_missing_close=0.00% pct_missing_value=0.00%
2026-04-11 00:42:29 [INFO] Saved 1306 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\date=2026-04-11\ACB.parquet
2026-04-11 00:42:29 [INFO] [2/50] Fetching BCM...
2026-04-11 00:42:40 [INFO] BCM — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-04-11 00:42:40 [INFO] BCM | src_used=VCI rows=1306 min_date=2021-01-12 max_date=2026-04-10 pct_missing_close=0.00% pct_missing_value=0.00%
2026-04-11 00:42:40 

(50,
 [('ACB',
   'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\date=2026-04-11\\ACB.parquet'),
  ('BCM',
   'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\date=2026-04-11\\BCM.parquet'),
  ('BID',
   'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\date=2026-04-11\\BID.parquet')])

In [4]:
# 2) Chi so VNINDEX/VN30/HNX...
index_outputs = ingest_indices(cfg)
index_outputs

2026-04-11 00:52:16 [INFO] [1/5] Fetching index VNINDEX...


2026-04-11 00:52:28 [INFO] VNINDEX — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-04-11 00:52:28 [INFO] VNINDEX | src_used=VCI rows=1306 min_date=2021-01-12 max_date=2026-04-10 pct_missing_close=0.00% pct_missing_value=0.00%
2026-04-11 00:52:28 [INFO] Saved 1306 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\date=2026-04-11\VNINDEX.parquet
2026-04-11 00:52:28 [INFO] [2/5] Fetching index VN30...
2026-04-11 00:52:40 [INFO] VN30 — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-04-11 00:52:40 [INFO] VN30 | src_used=VCI rows=1306 min_date=2021-01-12 max_date=2026-04-10 pct_missing_close=0.00% pct_missing_value=0.00%
2026-04-11 00:52:40 [INFO] Saved 1306 rows to C:\U

{'VNINDEX': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-04-11\\VNINDEX.parquet',
 'VN30': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-04-11\\VN30.parquet',
 'HNXINDEX': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-04-11\\HNXINDEX.parquet',
 'HNX30': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-04-11\\HNX30.parquet',
 'UPCOMINDEX': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-04-11\\UPCOMINDEX.parquet'}

In [5]:
# 3) Listing
listing_file = ingest_listing(cfg)
listing_file

2026-04-11 00:53:28 [INFO] symbols_by_exchange: 3315 mã (có sàn) từ VCI
2026-04-11 00:53:28 [INFO] Lọc stock: 3315 -> 1739 dòng
2026-04-11 00:53:28 [INFO] ✓ Đã ghi 1739 dòng -> C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet


'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\listing\\master\\listing.parquet'

In [6]:
# 4) Company overview (master append)
company_file = ingest_company_overview(cfg)
company_file

2026-04-11 00:53:28 [INFO] [1/50] Đang tải company ACB...
2026-04-11 00:53:40 [INFO] [2/50] Đang tải company BCM...
2026-04-11 00:53:52 [INFO] [3/50] Đang tải company BID...
2026-04-11 00:54:04 [INFO] [4/50] Đang tải company BVH...
2026-04-11 00:54:16 [INFO] [5/50] Đang tải company CTG...
2026-04-11 00:54:28 [INFO] [6/50] Đang tải company FPT...
2026-04-11 00:54:40 [INFO] [7/50] Đang tải company GAS...
2026-04-11 00:54:52 [INFO] [8/50] Đang tải company GVR...
2026-04-11 00:55:04 [INFO] [9/50] Đang tải company HDB...
2026-04-11 00:55:16 [INFO] [10/50] Đang tải company HPG...
2026-04-11 00:55:28 [INFO] [11/50] Đang tải company MBB...
2026-04-11 00:55:40 [INFO] [12/50] Đang tải company MSN...
2026-04-11 00:55:52 [INFO] [13/50] Đang tải company MWG...
2026-04-11 00:56:04 [INFO] [14/50] Đang tải company PLX...
2026-04-11 00:56:16 [INFO] [15/50] Đang tải company POW...
2026-04-11 00:56:29 [INFO] [16/50] Đang tải company SAB...
2026-04-11 00:56:40 [INFO] [17/50] Đang tải company SHB...
2026-0

'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\company\\master\\company_overview.parquet'

In [7]:
# 5) Financial ratio
ratio_outputs = ingest_financial_ratio(cfg)
len(ratio_outputs), list(ratio_outputs.items())[:3]

2026-04-11 01:03:28 [INFO] [1/50] Đang tải financial ratio ACB...


2026-04-11 01:03:42 [INFO] Saved 53 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\financial_ratio\date=2026-04-11\ACB.parquet
2026-04-11 01:03:42 [INFO] ✓ ratio ACB: 53 dòng (src=vci, period=quarter)
2026-04-11 01:03:42 [INFO] [2/50] Đang tải financial ratio BCM...
2026-04-11 01:03:53 [INFO] Saved 32 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\financial_ratio\date=2026-04-11\BCM.parquet
2026-04-11 01:03:53 [INFO] ✓ ratio BCM: 32 dòng (src=vci, period=quarter)
2026-04-11 01:03:54 [INFO] [3/50] Đang tải financial ratio BID...
2026-04-11 01:04:05 [INFO] Saved 52 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\financial_ratio\date=2026-04-11\BID.parquet
2026-04-11 01:04:05 [INFO] ✓ ratio BID: 52 dòng (src=vci, period=quarter)
2026-04-11 01:04:06 [INFO] [4/50] Đang tải financial ratio BVH...
2026-04-11 01:04:18 [INFO] Saved 52 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeli

(50,
 [('ACB',
   'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\financial_ratio\\date=2026-04-11\\ACB.parquet'),
  ('BCM',
   'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\financial_ratio\\date=2026-04-11\\BCM.parquet'),
  ('BID',
   'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\financial_ratio\\date=2026-04-11\\BID.parquet')])

In [8]:
# 6) Price board snapshot
price_board_file = ingest_price_board(cfg)
price_board_file

2026-04-11 01:13:40 [INFO] Saved 50 rows to C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price_board\date=2026-04-11\PRICE_BOARD_SNAPSHOT.parquet


'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price_board\\date=2026-04-11\\PRICE_BOARD_SNAPSHOT.parquet'

In [9]:
# 7) Tong ket du lieu theo tung category
from pathlib import Path
import pandas as pd

base = cfg.data_lake_root
today = cfg.run_date

summary = []
checks = {
    "price": base / "price" / f"date={today}",
    "index": base / "index" / f"date={today}",
    "financial_ratio": base / "financial_ratio" / f"date={today}",
    "price_board": base / "price_board" / f"date={today}",
    "listing": base / "listing" / "master" / "listing.parquet",
    "company": base / "company" / "master" / "company_overview.parquet",
}

for category, path in checks.items():
    if path.is_dir():
        files = sorted(path.glob("*.parquet"))
        row_count = 0
        sample_file = ""
        if files:
            sample_file = files[0].name
            try:
                row_count = int(pd.read_parquet(files[0]).shape[0])
            except Exception:
                row_count = -1
        summary.append({
            "category": category,
            "status": "OK" if files else "MISSING",
            "file_count": len(files),
            "sample_file": sample_file,
            "sample_rows": row_count,
        })
    else:
        exists = path.exists()
        rows = 0
        if exists:
            try:
                rows = int(pd.read_parquet(path).shape[0])
            except Exception:
                rows = -1
        summary.append({
            "category": category,
            "status": "OK" if exists else "MISSING",
            "file_count": 1 if exists else 0,
            "sample_file": path.name if exists else "",
            "sample_rows": rows,
        })

summary_df = pd.DataFrame(summary)
display(summary_df)

,category,status,file_count,sample_file,sample_rows
0,price,OK,50,ACB.parquet,1306
1,index,OK,5,HNX30.parquet,1306
2,financial_ratio,OK,50,ACB.parquet,53
3,price_board,OK,1,PRICE_BOARD_SNAPSHOT.parquet,50
4,listing,OK,1,listing.parquet,1739
5,company,OK,1,company_overview.parquet,50


In [10]:
# results = run_structure_ingestion_pipeline(cfg)
# list(results.keys())